# 3. CNN - 특징 감지하기

# 3.0 GPU

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


필터 = 합성곱 = convolution

## 3.1 합성곱

데이터 준비

In [ ]:
# Define transformations for the dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,))
])

# 데이터셋
train_dataset = datasets.FashionMNIST(root="data", train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root="data", train=False, download=True, transform=transform)

# 데이터 로더
train_loader = DataLoader(train_dataset, batch_size=1000, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

모델 클래스 정의

In [ ]:
class FashionCNN(nn.Module):
    def __init__(self):
        super(FashionCNN, self).__init__()
        self.layer1 = nn.Sequential(
            # in (1,28,28) -> out (64,28,28)
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            # in (64,28,28) -> out (64,14,14)
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.layer2 = nn.Sequential(
            # in (64,14,14) -> out (64,12,12)
            nn.Conv2d(64, 64, kernel_size=3),
            nn.ReLU(),
            # in (64,12,12) -> out (64,6,6)
            nn.MaxPool2d(2)
        )
        self.fc1 = nn.Linear(64*6*6, 128)
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        # contiguous 등의 차이가 있지만, 일단 reshape와 같다고 보면 될 듯...배치 외에는 한 줄로 편다...
        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        out = self.fc2(out)
        return out

훈련 함수 정의

In [ ]:
def train(model, loader, device, optimizer, criterion, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
    print(f"Train Epoch: {epoch} -- Loss: {loss.item():.6f}")

def test(model, loader, device):
    model.eval()
    with torch.no_grad():
        correct, total = 0, 0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            # 인덱스, 최댓값 반환, dim=1
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        print(f"Accuracy of the network on the 10,000 test images: {100 * correct / total}%")

모델, 손실함수, 최적화 인스턴스 만들기

In [10]:
model = FashionCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

학습하고 테스트

In [11]:
iters = 51
for epoch in range(1, iters):
    train(model, train_loader, device, optimizer, criterion, epoch)

test(model, test_loader, device)

Train Epoch: 1 -- Loss: 0.419149
Train Epoch: 2 -- Loss: 0.383603
Train Epoch: 3 -- Loss: 0.352774
Train Epoch: 4 -- Loss: 0.324733
Train Epoch: 5 -- Loss: 0.269703
Train Epoch: 6 -- Loss: 0.300091
Train Epoch: 7 -- Loss: 0.285332
Train Epoch: 8 -- Loss: 0.283477
Train Epoch: 9 -- Loss: 0.245766
Train Epoch: 10 -- Loss: 0.236835
Train Epoch: 11 -- Loss: 0.225352
Train Epoch: 12 -- Loss: 0.181767
Train Epoch: 13 -- Loss: 0.180185
Train Epoch: 14 -- Loss: 0.187859
Train Epoch: 15 -- Loss: 0.174768
Train Epoch: 16 -- Loss: 0.173997
Train Epoch: 17 -- Loss: 0.175829
Train Epoch: 18 -- Loss: 0.183645
Train Epoch: 19 -- Loss: 0.178289
Train Epoch: 20 -- Loss: 0.178777
Train Epoch: 21 -- Loss: 0.124586
Train Epoch: 22 -- Loss: 0.134225
Train Epoch: 23 -- Loss: 0.166245
Train Epoch: 24 -- Loss: 0.133708
Train Epoch: 25 -- Loss: 0.164481
Train Epoch: 26 -- Loss: 0.116914
Train Epoch: 27 -- Loss: 0.142308
Train Epoch: 28 -- Loss: 0.115587
Train Epoch: 29 -- Loss: 0.108935
Train Epoch: 30 -- Loss